# Notebook 03: ML Benchmarking — CRC Metabolome & Microbial Producer Analysis (v2.1)

**Scope: `YACHIDA-CRC-2019`, primary analysis set only** (n=277: Healthy + Early_CRC +
Advanced_CRC). Sinha and Kim are genus-level and are used for external validation in NB05,
not here. MP is excluded from the study; PostSurgery (HS) was split out in NB02, so the
pickle this notebook reads contains primary samples only.

## Pipeline

1. **Targets** = the union of
   - **robust DA metabolites** — FDR-significant in *both* the unadjusted rank test and the
     confounder-adjusted model, with the same direction (NB02's `Robust` column), from
     `Healthy vs Early_CRC` and `Healthy vs Advanced_CRC`; up to 8 elevated + 7 depleted per
     comparison. There is no nominal-p fallback: a comparison with few robust hits
     contributes few targets, and that is reported rather than padded.
   - **the 7 pre-specified polyamines** — the study's a-priori hypothesis targets, analysed
     whether or not they are differentially abundant, labelled `prespecified_polyamine` and
     never described as "dysregulated".
2. Visualise abundance differences (boxplots + cross-stage comparison).
3. Benchmark one XGBoost classifier per target (species CLR + clinical confounders →
   above/below-median metabolite level):
   - **Performance** = `AUC_nested`: hyperparameters tuned by `RandomizedSearchCV` *inside*
     each outer fold (5 outer folds, 10 candidates, 3 inner folds), scored on the untouched
     outer test fold.
   - **Significance** = permutation test with **pre-specified, data-independent**
     hyperparameters (`FIXED_PARAMS`) for both observed and label-shuffled scores on the
     same folds, BH-corrected across targets (`Perm_qvalue`).
4. Producer analysis for the pre-specified polyamines: out-of-fold SHAP from the same
   fixed-parameter model, with a **direction** per species (Spearman between a species'
   value and its SHAP value) and bootstrap rank CIs.

**Prerequisite:** run `01_preprocessing.ipynb` and `02_analysis.ipynb` first.

---

## Design notes (what was wrong before, and what this does instead)

1. **No disease-status proxy among the features.** Tumour location is "-" for every
   non-cancer sample, so on a Healthy+CRC set a tumour-site dummy is an exact disease
   indicator. `utils.build_confounder_matrix()` returns Age, BMI, Brinkman.Index, Alcohol and
   Gender only, and *raises* if tumour location is requested for a set containing non-cancer
   samples. Measured cost of the old leak: NB05's stage-1 accuracy fell 0.63 → 0.39 once it
   was removed.
2. **Confounders adjust the target; they are not features** (2026-10-20). Each target is
   residualised on them before its median split, so every test asks whether the species
   predict the metabolite *given* the confounders. They are imputed once from the
   confounders alone (no target, no species). With them as features, the permutation test
   could not separate their contribution from the species'.
3. **No oversampling.** A median split gives balanced classes by construction, so
   synthetic-minority oversampling only added noise. Targets whose minority class has
   < 20 samples (ties at the median) are skipped and logged.
4. **Tuning is nested.** The previous version tuned on all samples and then cross-validated
   on those same samples, which inflates AUC. The permutation test uses fixed hyperparameters
   so observed and null scores come from an identical procedure (the test is exact). The
   p-values are sequential (Besag–Clifford): a target stops at its 10th exceedance, otherwise
   it runs to `N_PERM` = max(1000, 20 × targets), so one target can pass BH among all of
   them; a fixed 100 permutations floored p at 1/101 and q near m/101.
5. **OOF SHAP is the producer ranking, and it is signed.** `Direction` = sign of the Spearman
   ρ between a species' abundance and its SHAP contribution to "high metabolite". Only
   positively associated species are candidate producers downstream (NB08/NB09) — several
   top-|SHAP| species are *negatively* associated with their polyamine, and the old pipeline
   would have called those producers.
6. **No cross-notebook evidence triangulation here.** That lives at the end of
   `09_pathway_network.ipynb`, and it starts from NB02's partial correlations rather than
   from this notebook's SHAP, so SHAP is not both the candidate generator and its own
   evidence.
7. **Sign conventions** (NB02 `_v2.csv` tables): `MeanDiff = mean(group2) − mean(group1)` and
   `Coef` (adjusted model) share the same sign — **> 0 means higher in the case group** —
   while `CliffsD` carries the **opposite** sign by construction.

In [ ]:
import sys, pickle, re, warnings
sys.path.insert(0, '.')
# Scoped, not blanket: a bare ignore muted convergence, numerical and
# missing-glyph warnings in this notebook.
warnings.filterwarnings('ignore', category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

from sklearn.experimental import enable_iterative_imputer  # noqa: F401 (must precede IterativeImputer import)
from sklearn.impute import IterativeImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from joblib import Parallel, delayed
from sklearn.model_selection import (
    StratifiedKFold, RepeatedStratifiedKFold, RandomizedSearchCV,
)
from sklearn.metrics import roc_curve, auc
from scipy.stats import spearmanr, rankdata
from statsmodels.stats.multitest import multipletests
from joblib import parallel_backend
import os, time

try:
    from xgboost import XGBClassifier
except ImportError as e:
    raise ImportError(
        "xgboost is required: the v2 pipeline collapses the old 5-classifier "
        "zoo to a single tuned XGBoost model (see markdown above). "
        "Install with `pip install xgboost`."
    ) from e

try:
    import shap
except ImportError as e:
    raise ImportError(
        "shap is required for the producer (Section 3) analysis. "
        "Install with `pip install shap`."
    ) from e

import utils
import viz
from utils import (RESULTS_DIR, DATA_DIR, POLYAMINE_KEGG, FDR_THRESHOLD,
                   STAGE_GROUP_ORDER,
                   # v2.2 pre-specified target class (see CONTRACTS.md section 5b)
                   DECARBOXYLATION_TARGET_KEGG, DECARBOXYLATION_SUBSTRATE_PRODUCT)

for sub in ['figures/ml', 'tables', 'intermediate', 'logs']:
    (RESULTS_DIR / sub).mkdir(parents=True, exist_ok=True)

# One shared theme (see viz.py). This notebook used dpi=110 and
# set_context('paper', font_scale=1.2) while NB01/02/06 used dpi=100 and no context, so
# identical fontsize= arguments rendered at different physical sizes between notebooks.
viz.apply_theme()

CRC_DS         = 'YACHIDA-CRC-2019'
# Grouped stage scheme (utils.STAGE_GROUP_MAP): the same groups NB02's robust-hit
# tables are built on, so target selection and modelling use one definition.
GROUP_EARLY    = 'Early_CRC'
GROUP_ADVANCED = 'Advanced_CRC'

print('xgboost and shap available.')
print(f'Stage groups in use: {STAGE_GROUP_ORDER}')
print('Setup complete.')

---
## Section 0: Load inputs


In [ ]:
# -- Load the feature-reduced pickle (YACHIDA, primary analysis set) -----------
with open(RESULTS_DIR / 'intermediate' / 'preprocessed_data_reduced_v2.pkl', 'rb') as f:
    _prep = pickle.load(f)

spc = _prep['transformed_species'][CRC_DS].copy()
mtb = _prep['transformed_mtb'][CRC_DS].copy()
meta_h = _prep['harmonized_meta'][CRC_DS].copy()
polyamine_columns_ds = _prep['polyamine_columns'][CRC_DS]

print(f"Pickle sample scope: {_prep.get('sample_scope', 'unspecified')}")
print(f'{CRC_DS}: {len(spc)} samples | {spc.shape[1]-1} species (reduced) | '
      f'{mtb.shape[1]-1} metabolites (reduced)')

# The pickle must already be primary-only; assert rather than re-filter, so a
# scope change upstream surfaces here instead of being silently absorbed.
_groups = set(meta_h['Stage.Group'].dropna())
assert _groups <= set(STAGE_GROUP_ORDER), \
    f'non-primary group(s) in the ML input: {_groups - set(STAGE_GROUP_ORDER)}'
assert not meta_h['Study.Group'].isin(['MP', 'HS']).any(), \
    'MP/HS must not reach the ML notebook'

print(f'\nStage.Group distribution:\n{meta_h["Stage.Group"].value_counts().reindex(STAGE_GROUP_ORDER).to_string()}')
print(f'\nFine-grained Study.Group:\n{meta_h["Study.Group"].value_counts().to_string()}')

In [ ]:
# -- Confounder matrix: Age, BMI, Brinkman.Index, Alcohol, Gender --------------
# From the reduced pickle's harmonized metadata (same samples as spc/mtb).
# These ADJUST every target (build_feature_matrix residualises it on them); they are
# not model features. As features, the permutation test could not separate the
# species' contribution from theirs.
# Tumour location is NOT requested: this is a Healthy+CRC set, and
# utils.build_confounder_matrix() refuses include_tumor_location=True on any set
# containing a non-cancer sample precisely so this cannot be done by accident.
conf_df = utils.build_confounder_matrix(meta_h)

IMPUTE_COLS = ['Age', 'BMI', 'Brinkman.Index', 'Alcohol']  # genuinely-missing-capable only
CONF_COLS   = [c for c in conf_df.columns if c != 'Sample']

# Guard against any disease-status proxy reaching the feature matrix.
_banned = [c for c in CONF_COLS
           if c.startswith(('TumorSite_', 'TumorLoc_'))
           or 'Stage' in c or c.startswith('Tumor')]
assert not _banned, f'disease-status proxy in features: {_banned}'

# And prove the guard is live rather than trusting it.
try:
    utils.build_confounder_matrix(meta_h, include_tumor_location=True)
    raise AssertionError('tumour-location leak guard is NOT active')
except ValueError:
    print('Leak guard verified: tumour location is refused on this Healthy+CRC set.')

print('Confounder matrix columns:', CONF_COLS)
print('\nMissingness (imputed from the confounders alone, before residualising targets):')
for col in IMPUTE_COLS:
    n_na = conf_df[col].isna().sum()
    print(f'  {col}: {n_na} / {len(conf_df)} missing')
print(conf_df[CONF_COLS].describe().round(2))

---
## Section 1: Target metabolites

### Sign conventions (NB02 tables)

| Column | Sign | Meaning |
|---|---|---|
| `MeanDiff` = mean(case) − mean(Healthy) | **> 0** | metabolite **higher in CRC** |
| `Coef` (confounder-adjusted model) | **> 0** | metabolite **higher in CRC** |
| `CliffsD` | **< 0** | metabolite **higher in CRC** (elevated) |
| `CliffsD` | **> 0** | metabolite **lower in CRC** (depleted) |

`CliffsD` therefore has the **opposite** sign to `MeanDiff`/`Coef`. That is a property of
`mannwhitneyu(Healthy, CRC)`: scipy's U counts (Healthy > CRC) pairs, so
`CliffsD = 2U/(n₁n₂) − 1` is positive when Healthy values rank higher.

### Selection

- **Robust DA targets.** Only metabolites with `Robust == True` in NB02's grouped tables —
  q < 0.05 in **both** the Mann-Whitney test and the confounder-adjusted linear model, with
  **matching direction**. Up to 8 most elevated (most negative `CliffsD`) + 7 most depleted
  per comparison, from `Healthy vs Early_CRC` and `Healthy vs Advanced_CRC`.

  Requiring both tests matters here specifically: smoking is strongly imbalanced across
  these groups (median Brinkman index 100 in Healthy vs 196–390 in CRC), so a metabolite
  that passes only the unadjusted test may be tracking smoking rather than cancer. Those
  `mw_only` features are counted and reported below, not silently dropped — and not
  modelled.

- **Pre-specified polyamines.** Every polyamine column present, analysed as an a-priori
  hypothesis target regardless of DA significance (`Target_type = prespecified_polyamine`),
  with its DA q-values reported alongside. This is what keeps the study's central question
  answerable even if no polyamine survives FDR.

In [ ]:
# -- Load NB02's grouped, robust-annotated DA table ---------------------------
# v2.1: one tidy table (Dataset | Comparison | Scheme | Assay | Feature | ...) replaces
# the six per-comparison CSVs. Sliced by Comparison + Assay rather than by filename.
_DA_TIDY = RESULTS_DIR / 'tables' / 'differential_abundance_v2.csv'


def _load_da(group):
    if not _DA_TIDY.exists():
        raise FileNotFoundError(
            f'{_DA_TIDY.name} not found. NB02 Step 6B writes it; run 02_analysis.ipynb '
            'first. (Before v2.1 this read per-comparison '
            f'{CRC_DS}_Healthy_vs_<group>_metabolites_DA_v2.csv files.)')
    da = pd.read_csv(_DA_TIDY)
    for col in ('Robust', 'Agreement', 'QValue_adj', 'Coef_adj'):
        if col not in da.columns:
            raise KeyError(
                f'{_DA_TIDY.name} has no {col!r} column -- it was written by an older NB02 '
                'that did not run the confounder-adjusted model. Re-run 02_analysis.ipynb.')
    df = da[(da['Dataset'] == CRC_DS)
            & (da['Comparison'] == f'Healthy_vs_{group}')
            & (da['Assay'] == 'metabolites')].reset_index(drop=True)
    if df.empty:
        raise ValueError(
            f'{_DA_TIDY.name} has no rows for Healthy_vs_{group} metabolites in {CRC_DS}. '
            f'Comparisons present: {sorted(da["Comparison"].unique())}')
    return df

early_da    = _load_da(GROUP_EARLY)
advanced_da = _load_da(GROUP_ADVANCED)

POLYAMINE_COL_NAMES = set(polyamine_columns_ds.values())
KEGG_TO_NAME = {v: POLYAMINE_KEGG.get(k, k) for k, v in polyamine_columns_ds.items()}


def short_name(feat):
    """Strip the KEGG prefix for display."""
    parts = feat.split('_', 1)
    return parts[1] if len(parts) > 1 else feat


def select_robust_balanced(da_df, n_up=8, n_down=7, label=''):
    """Robust-only selection: among features significant in BOTH tests with the same
    direction, take the n_up most elevated and n_down most depleted (by CliffsD).

    No nominal-p and no MW-only fallback: a comparison with few robust hits
    contributes few targets.
    """
    rob = da_df[da_df['Robust'] == True].copy()        # noqa: E712 (CSV may read as object)
    up   = rob[rob['CliffsD'] < 0].nsmallest(n_up, 'CliffsD')
    down = rob[rob['CliffsD'] > 0].nlargest(n_down, 'CliffsD')
    sel = pd.concat([up, down]).reset_index(drop=True)
    sel['is_polyamine'] = sel['Feature'].isin(POLYAMINE_COL_NAMES)
    sel['direction'] = sel['CliffsD'].apply(
        lambda d: 'Elevated in CRC' if d < 0 else 'Depleted in CRC')

    if label:
        agree = da_df['Agreement'].value_counts().to_dict()
        n_mw  = int((da_df['QValue'] < FDR_THRESHOLD).sum())
        n_rob = int((da_df['Robust'] == True).sum())     # noqa: E712
        print(f'{label}:')
        print(f'    unadjusted q<0.05 : {n_mw}')
        print(f'    robust (both+dir) : {n_rob}')
        print(f'    dropped by adjustment (mw_only): {agree.get("mw_only", 0)}'
              f'  | revealed by adjustment (lm_only): {agree.get("lm_only", 0)}'
              f'  | direction conflict: {agree.get("both_q_opposite_dir", 0)}')
        print(f'    selected as targets: {len(sel)} '
              f'({int((sel["CliffsD"] < 0).sum())} elevated + '
              f'{int((sel["CliffsD"] > 0).sum())} depleted), '
              f'{int(sel["is_polyamine"].sum())} polyamine(s)')
    return sel


print('=== Robust DA target selection (grouped stage scheme) ===')
top15_early    = select_robust_balanced(early_da,    label=f'Healthy vs {GROUP_EARLY}')
top15_advanced = select_robust_balanced(advanced_da, label=f'Healthy vs {GROUP_ADVANCED}')

top15_early_set    = set(top15_early['Feature'])
top15_advanced_set = set(top15_advanced['Feature'])

# -- Pre-specified polyamine targets (a-priori hypothesis; not DA-selected) ----
def _da_row(da_df, feat):
    r = da_df[da_df['Feature'] == feat]
    if r.empty:
        return (np.nan, np.nan, np.nan, 'not_tested')
    return (float(r['QValue'].iloc[0]), float(r['CliffsD'].iloc[0]),
            float(r['QValue_adj'].iloc[0]) if pd.notna(r['QValue_adj'].iloc[0]) else np.nan,
            str(r['Agreement'].iloc[0]))

pa_target_rows = []
for kegg_id, col in polyamine_columns_ds.items():
    if col not in mtb.columns:
        print(f'  NOTE: polyamine {POLYAMINE_KEGG.get(kegg_id)} ({col}) absent from the '
              f'reduced matrix -- cannot be modelled')
        continue
    q_e, d_e, qadj_e, ag_e = _da_row(early_da, col)
    q_a, d_a, qadj_a, ag_a = _da_row(advanced_da, col)
    pa_target_rows.append({
        'Feature': col, 'KEGG': kegg_id, 'PA_name': POLYAMINE_KEGG.get(kegg_id, kegg_id),
        'QValue_early': q_e, 'CliffsD_early': d_e, 'QValue_adj_early': qadj_e,
        'Agreement_early': ag_e,
        'QValue_advanced': q_a, 'CliffsD_advanced': d_a, 'QValue_adj_advanced': qadj_a,
        'Agreement_advanced': ag_a,
        'Target_type': 'prespecified_polyamine'})
pa_targets = pd.DataFrame(pa_target_rows)

print(f'\nPre-specified polyamine targets: {len(pa_targets)}')
if not pa_targets.empty:
    print(pa_targets[['PA_name', 'QValue_early', 'QValue_adj_early', 'Agreement_early',
                      'QValue_advanced', 'QValue_adj_advanced', 'Agreement_advanced']]
          .round(4).to_string(index=False))

print(f'\n=== Robust targets: Healthy vs {GROUP_EARLY} ===')
print(top15_early[['Feature', 'CliffsD', 'QValue', 'QValue_adj', 'direction',
                   'is_polyamine']].to_string(index=False) if not top15_early.empty
      else '  (none -- this comparison contributes no robust DA targets)')
print(f'\n=== Robust targets: Healthy vs {GROUP_ADVANCED} ===')
print(top15_advanced[['Feature', 'CliffsD', 'QValue', 'QValue_adj', 'direction',
                      'is_polyamine']].to_string(index=False) if not top15_advanced.empty
      else '  (none -- this comparison contributes no robust DA targets)')

In [ ]:
# ── Pre-specified target class: microbial amino-acid decarboxylation (v2.2) ──
# Chosen from an ENZYME FAMILY in the literature, not from any statistic computed on this
# cohort. That independence is load-bearing: Section 2 asks whether the SPECIES predict
# each target against a label-permutation null, so selecting targets BECAUSE species
# correlate with them would make the test reject by construction and reduce the
# predictive evidence source in NB08/NB09 to a restatement of the statistical one.
# checks.py E14 fails if this notebook reads any within-cohort species x metabolite
# correlation table.
#
# The class subsumes the polyamine hypothesis rather than discarding it: putrescine,
# cadaverine and agmatine are themselves decarboxylation products (of ornithine, lysine
# and arginine). What it adds is the rest of the family -- the aromatic and basic amino
# acids -- and, for every member, the SUBSTRATE as well as the product. A species driving
# a decarboxylation should deplete the substrate and elevate the product, so the pair is
# a directional signature rather than a bare association.
_mtb_by_kegg = {}
for _c in mtb.columns:
    if _c == 'Sample':
        continue
    _m = re.match(r'(C\d{5})_', _c)
    if _m:
        _mtb_by_kegg[_m.group(1)] = _c

print('=== Pre-specified class: amino-acid decarboxylation (substrate + product) ===')
print(f'{"substrate":>16} {"":>4} {"product":>16}   enzyme')
for _sub, (_prod, _enz, _ec) in DECARBOXYLATION_SUBSTRATE_PRODUCT.items():
    _sn = DECARBOXYLATION_TARGET_KEGG[_sub][0]
    _pn = DECARBOXYLATION_TARGET_KEGG[_prod][0]
    _ok = '->' if (_sub in _mtb_by_kegg and _prod in _mtb_by_kegg) else ' x'
    print(f'{_sn:>16} {_ok:>4} {_pn:>16}   {_enz} (EC {_ec})')

decarb_rows = []
for _kegg, (_name, _role) in DECARBOXYLATION_TARGET_KEGG.items():
    _col = _mtb_by_kegg.get(_kegg)
    if _col is None:
        print(f'  NOTE: {_name} ({_kegg}) is not measured in the reduced matrix '
              f'-- cannot be modelled')
        continue
    q_e, d_e, qadj_e, ag_e = _da_row(early_da, _col)
    q_a, d_a, qadj_a, ag_a = _da_row(advanced_da, _col)
    decarb_rows.append({
        'Feature': _col, 'KEGG': _kegg, 'Compound': _name, 'Class_role': _role,
        'QValue_early': q_e, 'CliffsD_early': d_e, 'QValue_adj_early': qadj_e,
        'Agreement_early': ag_e,
        'QValue_advanced': q_a, 'CliffsD_advanced': d_a, 'QValue_adj_advanced': qadj_a,
        'Agreement_advanced': ag_a,
        'Target_type': 'prespecified_decarboxylation'})
decarb_targets = pd.DataFrame(decarb_rows)

# What this class adds beyond the robust-DA and polyamine lists already selected above.
_existing = set(top15_early['Feature']) | set(top15_advanced['Feature']) | (
    set(pa_targets['Feature']) if not pa_targets.empty else set())
decarb_targets['Already_selected'] = decarb_targets['Feature'].isin(_existing)
decarb_targets.to_csv(
    RESULTS_DIR / 'tables' / 'nb03_prespecified_decarboxylation_targets_v2.csv', index=False)

print(f'\nClass members measured here: {len(decarb_targets)} of '
      f'{len(DECARBOXYLATION_TARGET_KEGG)}')
print(f'  already selected by robust DA / the polyamine list: '
      f'{int(decarb_targets["Already_selected"].sum())}')
print(f'  NEW targets contributed by this class            : '
      f'{int((~decarb_targets["Already_selected"]).sum())}')
if not decarb_targets.empty:
    print()
    print(decarb_targets[['Compound', 'Feature', 'Class_role', 'QValue_adj_early',
                          'QValue_adj_advanced', 'Already_selected']]
          .round(4).to_string(index=False))
print('\nSaved: nb03_prespecified_decarboxylation_targets_v2.csv')

In [ ]:
# -- Boxplots: robust targets, Healthy vs Early CRC ---------------------------
stage_mask_early = meta_h['Stage.Group'].isin(['Healthy', GROUP_EARLY])
meta_e   = meta_h[stage_mask_early].copy()
mtb_e    = mtb[mtb['Sample'].isin(meta_e['Sample'])]
merged_e = meta_e.merge(mtb_e, on='Sample')

if top15_early.empty:
    print(f'No robust DA targets for Healthy vs {GROUP_EARLY} -- no boxplot panel.')
else:
    n = len(top15_early)
    ncols = min(5, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3.5), squeeze=False)
    axes_flat = axes.flatten()

    for i, (_, row) in enumerate(top15_early.iterrows()):
        ax = axes_flat[i]
        feat = row['Feature']
        vals_h = merged_e[merged_e['Stage.Group'] == 'Healthy'][feat].dropna()
        vals_c = merged_e[merged_e['Stage.Group'] == GROUP_EARLY][feat].dropna()

        crc_color = '#d63031' if row['CliffsD'] < 0 else '#0984e3'
        bplot = ax.boxplot([vals_h, vals_c], patch_artist=True, widths=0.5,
                            showfliers=False, medianprops=dict(color='black', linewidth=1.5))
        bplot['boxes'][0].set_facecolor('#dfe6e9'); bplot['boxes'][0].set_alpha(0.85)
        bplot['boxes'][1].set_facecolor(crc_color); bplot['boxes'][1].set_alpha(0.85)

        ax.set_xticklabels(['Healthy', 'Early\nCRC'], fontsize=9)
        ax.set_title(short_name(feat), fontsize=8.5, fontweight='bold',
                     color='#c0392b' if row['is_polyamine'] else 'black')
        ax.set_ylabel('log₁₀(abund.)', fontsize=7.5)
        ax.text(0.97, 0.97, f"q={row['QValue']:.3f}\nq_adj={row['QValue_adj']:.3f}",
                transform=ax.transAxes, ha='right', va='top', fontsize=6.5, style='italic')
        dir_label = 'Elevated' if row['CliffsD'] < 0 else 'Depleted'
        ax.text(0.03, 0.97, dir_label, transform=ax.transAxes,
                ha='left', va='top', fontsize=7, fontweight='bold', color=crc_color)
        if row['is_polyamine']:
            for spine in ax.spines.values():
                spine.set_edgecolor('#c0392b'); spine.set_linewidth(2)

    for j in range(n, len(axes_flat)):
        axes_flat[j].set_visible(False)

    fig.suptitle('Robust DA metabolites — Early CRC vs Healthy\n'
                 '(significant in BOTH the rank test and the confounder-adjusted model)\n'
                 'Red box = elevated in CRC | Blue box = depleted | Red border = polyamine',
                 fontsize=11, fontweight='bold')
    plt.tight_layout()
    out = RESULTS_DIR / 'figures' / 'ml' / 'top15_metabolites_early_CRC_v2.png'
    viz.save_figure(plt.gcf(), out)
    plt.show(); plt.close()
    print(f'Saved: {out}')

### Interpretation: early-stage metabolite changes

Only metabolites robust to confounder adjustment are used as DA targets. If few or none
survive for `Healthy vs Early_CRC`, this comparison contributes few or no targets — reported
as is, never padded with nominally significant metabolites. The counts printed above show
how many unadjusted hits were lost to adjustment (`mw_only`), which is the quantity of
interest: in this cohort smoking differs 2–4× across groups, so unadjusted early-stage
"signal" is the most likely place for confounding to masquerade as biology.

Polyamines are analysed regardless, as pre-specified targets.

In [ ]:
# -- Boxplots: robust targets, Healthy vs Advanced CRC ------------------------
stage_mask_adv = meta_h['Stage.Group'].isin(['Healthy', GROUP_ADVANCED])
meta_a   = meta_h[stage_mask_adv].copy()
mtb_a    = mtb[mtb['Sample'].isin(meta_a['Sample'])]
merged_a = meta_a.merge(mtb_a, on='Sample')

if top15_advanced.empty:
    print(f'No robust DA targets for Healthy vs {GROUP_ADVANCED} -- no boxplot panel.')
else:
    n = len(top15_advanced)
    ncols = min(5, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3.5), squeeze=False)
    axes_flat = axes.flatten()

    for i, (_, row) in enumerate(top15_advanced.iterrows()):
        ax = axes_flat[i]
        feat = row['Feature']
        vals_h = merged_a[merged_a['Stage.Group'] == 'Healthy'][feat].dropna()
        vals_c = merged_a[merged_a['Stage.Group'] == GROUP_ADVANCED][feat].dropna()

        crc_color = '#d63031' if row['CliffsD'] < 0 else '#0984e3'
        bplot = ax.boxplot([vals_h, vals_c], patch_artist=True, widths=0.5,
                            showfliers=False, medianprops=dict(color='black', linewidth=1.5))
        bplot['boxes'][0].set_facecolor('#dfe6e9'); bplot['boxes'][0].set_alpha(0.85)
        bplot['boxes'][1].set_facecolor(crc_color); bplot['boxes'][1].set_alpha(0.85)

        ax.set_xticklabels(['Healthy', 'Adv.\nCRC'], fontsize=9)
        ax.set_title(short_name(feat), fontsize=8.5, fontweight='bold',
                     color='#c0392b' if row['is_polyamine'] else 'black')
        ax.set_ylabel('log₁₀(abund.)', fontsize=7.5)
        ax.text(0.97, 0.97, f"q={row['QValue']:.3f}\nq_adj={row['QValue_adj']:.3f}",
                transform=ax.transAxes, ha='right', va='top', fontsize=6.5, style='italic')
        dir_label = 'Elevated' if row['CliffsD'] < 0 else 'Depleted'
        ax.text(0.03, 0.97, dir_label, transform=ax.transAxes,
                ha='left', va='top', fontsize=7, fontweight='bold', color=crc_color)
        if row['is_polyamine']:
            for spine in ax.spines.values():
                spine.set_edgecolor('#c0392b'); spine.set_linewidth(2)

    for j in range(n, len(axes_flat)):
        axes_flat[j].set_visible(False)

    fig.suptitle('Robust DA metabolites — Advanced CRC vs Healthy\n'
                 '(significant in BOTH the rank test and the confounder-adjusted model)\n'
                 'Red box = elevated in CRC | Blue box = depleted | Red border = polyamine',
                 fontsize=11, fontweight='bold')
    plt.tight_layout()
    out = RESULTS_DIR / 'figures' / 'ml' / 'top15_metabolites_advanced_CRC_v2.png'
    viz.save_figure(plt.gcf(), out)
    plt.show(); plt.close()
    print(f'Saved: {out}')

In [ ]:
# -- Cross-stage comparison scatter -------------------------------------------
compare = (early_da[['Feature', 'CliffsD', 'Robust']]
           .rename(columns={'CliffsD': 'CD_early', 'Robust': 'Rob_early'})
           .merge(advanced_da[['Feature', 'CliffsD', 'Robust']]
                  .rename(columns={'CliffsD': 'CD_advanced', 'Robust': 'Rob_advanced'}),
                  on='Feature'))
compare['is_polyamine'] = compare['Feature'].isin(POLYAMINE_COL_NAMES)
compare['in_top15_e']   = compare['Feature'].isin(top15_early_set)
compare['in_top15_a']   = compare['Feature'].isin(top15_advanced_set)

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(compare['CD_early'], compare['CD_advanced'],
           c='#bdc3c7', s=25, alpha=0.6, zorder=1, label='Other metabolites')
mask_e = compare['in_top15_e'] & ~compare['is_polyamine']
ax.scatter(compare.loc[mask_e, 'CD_early'], compare.loc[mask_e, 'CD_advanced'],
           c='#3498db', s=60, alpha=0.9, zorder=2, label='Robust target (Early)')
mask_a = compare['in_top15_a'] & ~compare['is_polyamine']
ax.scatter(compare.loc[mask_a, 'CD_early'], compare.loc[mask_a, 'CD_advanced'],
           c='#e67e22', s=60, alpha=0.9, zorder=2, label='Robust target (Advanced)')
mask_pa = compare['is_polyamine']
ax.scatter(compare.loc[mask_pa, 'CD_early'], compare.loc[mask_pa, 'CD_advanced'],
           c='#c0392b', marker='*', s=200, zorder=3, label='Polyamines (pre-specified)')
for _, r in compare[mask_pa].iterrows():
    name = KEGG_TO_NAME.get(r['Feature'], short_name(r['Feature']))
    ax.annotate(name, (r['CD_early'], r['CD_advanced']), fontsize=8, color='#c0392b',
                fontweight='bold', xytext=(6, 4), textcoords='offset points')

ax.axhline(0, color='gray', lw=0.8, ls='--')
ax.axvline(0, color='gray', lw=0.8, ls='--')
ax.set_xlabel("Cliff's δ  (Early CRC vs Healthy)\n"
              "negative = elevated in CRC  |  positive = depleted in CRC", fontsize=11)
ax.set_ylabel("Cliff's δ  (Advanced CRC vs Healthy)\n"
              "negative = elevated in CRC  |  positive = depleted in CRC", fontsize=11)
ax.set_title("Metabolite dysregulation: Early vs Advanced CRC\n"
             "(coloured points are robust in that comparison)", fontsize=11, fontweight='bold')
ax.legend(loc='lower right', fontsize=9, framealpha=0.9)
plt.tight_layout()
out = RESULTS_DIR / 'figures' / 'ml' / 'metabolite_dysregulation_comparison_v2.png'
viz.save_figure(plt.gcf(), out)
plt.show(); plt.close()
print(f'Saved: {out}')

---
## Section 2: ML Benchmark — Microbial Source Prediction

For each target metabolite (robust DA targets + pre-specified polyamines):
- **Features**: species CLR abundances only. No tumour location, no stage indicator — see
  the leak guard in Section 0.
- **Target**: the metabolite adjusted for Age, BMI, Gender, Brinkman Index and Alcohol (OLS
  residual, `utils.residualise_on_confounders`), split at its median — high or low
  *relative to what the confounders predict*. The confounders used to be features, so the
  permutation test below asked whether the target was independent of the species **and**
  the confounders; the confounders alone predict putrescine at AUC 0.67, so a significant
  p-value said nothing about the species. Every p-value here tests the species **given**
  the confounders (exact for linear confounding; calibration in CONTRACTS.md, NB03).
  `R2_confounders`, `AUC_confounders_raw` and `AUC_confounders_adjusted` report what the
  confounders explain on their own.
- **Model**: XGBoost; `StandardScaler` is refit on each training fold.
- **`AUC_nested`** (performance): 5-fold outer CV; inside each training fold a
  `RandomizedSearchCV` (10 candidates, 3 inner folds, ROC-AUC) selects hyperparameters and
  the refit model is scored on the untouched outer test fold.
- **`Perm_pvalue` / `Perm_qvalue`** (significance): observed AUC of the fixed-parameter
  model (`AUC_fixed`; the p-value refers to it, not to `AUC_nested`) against label
  permutations on the same 5 folds, **sequential** (Besag–Clifford): a target stops at its
  10th exceedance (p = 10/L); otherwise it runs to `N_PERM` = max(1000, 20 × targets) and
  p = (x + 1)/(`N_PERM` + 1). Then BH across targets. A fixed 100 permutations put the
  smallest reachable q near m/101, so no target could pass BH on its own. Fixed parameters are what make the comparison exact — the null and the
  observed score come from an identical procedure.
- A timing probe prints the projected runtime before the loop (`NB03_N_PERM` overrides
  the cap `N_PERM`).

**What an AUC here does and does not mean.** The target is a median split of a *measured*
metabolite, so a high AUC says the species composition carries information about that
metabolite's level in this cohort. It is not evidence that those species produce it: that
requires direction (Section 3), genomic capability (NB08), and ideally flux (NB10), which is
why the final synthesis in NB09 counts those as separate evidence sources.

In [ ]:
# ── Pipeline building blocks ──────────────────────────────────────────────────

def _aligned_inputs(species_df, mtb_df, target_col, conf_df):
    """Species matrix, raw target and imputed confounders on one sample set and order."""
    merged = (species_df.merge(mtb_df[['Sample', target_col]], on='Sample')
                          .merge(conf_df, on='Sample'))
    feat_cols = [c for c in species_df.columns if c != 'Sample']
    assert not set(feat_cols) & set(CONF_COLS), 'a confounder reached the feature matrix'
    X = merged[feat_cols].to_numpy(dtype=np.float64)
    y_raw = merged[target_col].to_numpy(dtype=np.float64)
    # Imputed once, on all samples, from the confounders ALONE -- no target, no
    # species -- so the imputation carries no label information into the folds.
    Z = IterativeImputer(max_iter=10, random_state=42).fit_transform(
        merged[CONF_COLS].to_numpy(dtype=np.float64))
    return X, y_raw, Z, feat_cols, merged['Sample'].to_numpy()


def build_feature_matrix(species_df, mtb_df, target_col, conf_df):
    """Species-only features and the confounder-ADJUSTED target:
    returns (X, y_adj, feat_cols, sample_ids).

    y_adj is the OLS residual of the target on Age, BMI, sex, Brinkman index and
    alcohol (utils.residualise_on_confounders), so the model, its permutation test and
    its SHAP values see only what the species carry BEYOND the confounders. The
    confounders used to be features, so the label permutation tested 'target
    independent of species and confounders' -- which the confounders alone reject
    (putrescine AUC 0.67) -- and a significant p said nothing about the species.
    The TARGET is residualised, not the species: species residuals keep confounder
    information in their zero patterns and the trees recover it (CONTRACTS.md, NB03).
    """
    X, y_raw, Z, feat_cols, ids = _aligned_inputs(species_df, mtb_df, target_col, conf_df)
    return X, utils.residualise_on_confounders(y_raw, Z), feat_cols, ids


def confounder_view(species_df, mtb_df, target_col, conf_df):
    """(y_raw, Z) on exactly the samples, in the order, build_feature_matrix uses."""
    _, y_raw, Z, _, _ = _aligned_inputs(species_df, mtb_df, target_col, conf_df)
    return y_raw, Z


def binarize_target(y, threshold=None):
    """Binary: 1 = high (above median or given threshold)."""
    thresh = float(np.median(y)) if threshold is None else float(threshold)
    return (y > thresh).astype(int)


def make_preprocess_pipeline(random_state=42):
    """The y-INDEPENDENT step, refit per training fold: scaling. Confounder imputation
    left the pipeline with the confounders, which now adjust the target instead of
    being features. (random_state is kept for call-site compatibility.)"""
    return Pipeline([('scale', StandardScaler())])


# Pre-specified, data-independent hyperparameters (used for the permutation test and SHAP,
# so the tested/explained model is never tuned on the evaluation data).
FIXED_PARAMS = dict(n_estimators=200, max_depth=3, learning_rate=0.05,
                    subsample=0.8, colsample_bytree=0.8, min_child_weight=1)


def make_model(xgb_params=None, random_state=42, n_jobs=1):
    """XGBoost classifier (n_jobs=1: always fit inside an already-parallel loop)."""
    params = dict(FIXED_PARAMS)
    if xgb_params:
        params.update(xgb_params)
    # max_bin=64: ~3x faster per fit on this data than the default 256.
    return XGBClassifier(random_state=random_state, n_jobs=n_jobs, eval_metric='logloss',
                         tree_method='hist', max_bin=64, **params)


def make_pipeline(xgb_params=None, random_state=42, n_jobs=1):
    """Full impute -> scale -> XGBoost pipeline."""
    pre = make_preprocess_pipeline(random_state=random_state)
    return Pipeline(pre.steps + [('clf', make_model(xgb_params, random_state, n_jobs))])


XGB_PARAM_DIST = {
    'clf__n_estimators':     [100, 200, 300, 500],
    'clf__max_depth':        [2, 3, 4, 5, 6],
    'clf__learning_rate':    [0.01, 0.03, 0.05, 0.1],
    'clf__subsample':        [0.6, 0.8, 1.0],
    'clf__colsample_bytree': [0.6, 0.8, 1.0],
    'clf__min_child_weight': [1, 3, 5],
}


def nested_cv_auc(X, y_bin, n_outer=5, n_iter=10, inner=3, random_state=42, n_jobs=2):
    """Nested CV: hyperparameters tuned INSIDE each outer training fold, scored on the
    untouched outer test fold. Returns (fold_aucs, chosen_params_per_fold)."""
    outer = StratifiedKFold(n_splits=n_outer, shuffle=True, random_state=random_state)
    aucs, chosen = [], []
    for tr, te in outer.split(X, y_bin):
        search = RandomizedSearchCV(
            make_pipeline(random_state=random_state), XGB_PARAM_DIST, n_iter=n_iter,
            scoring='roc_auc', cv=StratifiedKFold(n_splits=inner, shuffle=True, random_state=random_state),
            random_state=random_state, n_jobs=n_jobs, refit=True,
        )
        # Threads, not processes: XGBoost releases the GIL while training, and worker
        # processes would each re-import the stack (the 1.5 GB server cap).
        with parallel_backend('threading', n_jobs=n_jobs):
            search.fit(X[tr], y_bin[tr])
        proba = search.predict_proba(X[te])[:, 1]
        fpr, tpr, _ = roc_curve(y_bin[te], proba)
        aucs.append(auc(fpr, tpr))
        chosen.append({k.replace('clf__', ''): v for k, v in search.best_params_.items()})
    return np.array(aucs), chosen


def build_fold_cache(X, y_bin, n_splits=5, random_state=42):
    """ONE fixed CV partition (sklearn permutation_test_score convention: fixed
    splitter, only labels shuffled); impute+scale fit once per fold on train indices."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    cache = []
    for train_idx, test_idx in skf.split(X, y_bin):
        pre = make_preprocess_pipeline(random_state=random_state)
        cache.append({'train_idx': train_idx, 'test_idx': test_idx,
                      'X_train': pre.fit_transform(X[train_idx]),
                      'X_test': pre.transform(X[test_idx])})
    return cache


def run_cv_auc_cached(fold_cache, y_bin, xgb_params=None, random_state=42, n_jobs=1):
    """Fit the classifier per cached fold; identical for observed and permuted y."""
    mean_fpr = np.linspace(0, 1, 100)
    tprs, fold_aucs = [], []
    for fold in fold_cache:
        model = make_model(xgb_params, random_state=random_state, n_jobs=n_jobs)
        model.fit(fold['X_train'], y_bin[fold['train_idx']])
        proba = model.predict_proba(fold['X_test'])[:, 1]
        fpr, tpr, _ = roc_curve(y_bin[fold['test_idx']], proba)
        fold_aucs.append(auc(fpr, tpr))
        tprs.append(np.interp(mean_fpr, fpr, tpr))
    mean_tpr = np.mean(tprs, axis=0)
    mean_tpr[0], mean_tpr[-1] = 0.0, 1.0
    return mean_fpr, mean_tpr, np.array(fold_aucs)


def _one_permutation_auc(fold_cache, y_bin, xgb_params, perm_seed, random_state):
    y_shuf = np.random.default_rng(perm_seed).permutation(y_bin)
    _, _, fold_aucs = run_cv_auc_cached(fold_cache, y_shuf, xgb_params,
                                         random_state=random_state, n_jobs=1)
    return float(fold_aucs.mean())


def run_permutation_null_cached(fold_cache, y_bin, xgb_params=None, n_perm=100,
                                 random_state=42, base_seed=1000, n_jobs=2):
    perm_aucs = Parallel(n_jobs=n_jobs, backend='threading')(
        delayed(_one_permutation_auc)(fold_cache, y_bin, xgb_params, base_seed + p, random_state)
        for p in range(n_perm)
    )
    return np.array(perm_aucs)


def extract_shap_class1(sv_raw):
    """Positive-class SHAP array (n_samples, n_features), SIGNED, regardless of the
    SHAP/XGBoost return format."""
    if isinstance(sv_raw, list):
        return np.asarray(sv_raw[1])
    sv_raw = np.asarray(sv_raw)
    if sv_raw.ndim == 3:
        return sv_raw[..., 1] if sv_raw.shape[-1] == 2 else sv_raw[1]
    return sv_raw


def species_short(name):
    """Extract a readable species name from a MetaPhlAn taxonomic string."""
    last = name.split('|')[-1]
    return last.replace('s__', '').replace('_', ' ').strip()


print('Pipeline helpers defined (nested CV, fixed-parameter permutation test, signed SHAP).')


In [ ]:
# -- Benchmark: nested-CV AUC + fixed-parameter permutation test -------------
da_features = list(dict.fromkeys(
    top15_early['Feature'].tolist() + top15_advanced['Feature'].tolist()))
pa_features = pa_targets['Feature'].tolist() if not pa_targets.empty else []
# v2.2: the pre-specified decarboxylation class (c06b). Both criteria are independent of
# any within-cohort species x metabolite statistic -- robust DA selects on the DISEASE
# axis, this class on an enzyme family -- so the permutation test below stays a valid
# test of "do the species predict this metabolite" (checks.py E14).
dc_features = decarb_targets['Feature'].tolist() if not decarb_targets.empty else []
target_metabolites = list(dict.fromkeys(da_features + pa_features + dc_features))
# A metabolite can qualify under more than one criterion (phenylalanine is both a
# robust DA hit and a decarboxylase substrate), so the union must stay
# deduplicated. A duplicate benchmarks the target twice, and bench_df.loc[target]
# then returns a Series -- which surfaces far away as "cannot convert the series
# to float" inside the SHAP loop rather than here.
assert len(target_metabolites) == len(set(target_metabolites)), \
    'duplicate target in the benchmark list'
print(f'Target metabolites: {len(target_metabolites)} '
      f'({len(da_features)} robust DA, {len(pa_features)} pre-specified polyamines, '
      f'{len(dc_features)} pre-specified decarboxylation class, '
      f'{len(set(dc_features) - set(da_features) - set(pa_features))} of those new)')
if not da_features:
    print('NOTE: no robust DA targets survived confounder adjustment in either comparison.\n'
          '      The benchmark therefore runs on the pre-specified polyamines alone, which\n'
          '      is a valid a-priori analysis -- not a fallback that was tuned to find hits.')

direction_map = {}
_sel_all = [d for d in (top15_early, top15_advanced) if not d.empty]
if _sel_all:
    for _, r in pd.concat(_sel_all).drop_duplicates('Feature').iterrows():
        direction_map[r['Feature']] = 'Elevated' if r['CliffsD'] < 0 else 'Depleted'

N_OUTER     = 5
N_ITER      = int(os.environ.get('NB03_N_ITER', 10))
N_INNER     = 3
PERM_H      = 10    # Besag-Clifford: a target's permutations stop at its 10th exceedance
# Outer parallelism. run_pipeline.py sets KI_N_JOBS to the physical core count and
# OMP_NUM_THREADS=1, so each fit is single-threaded and the cores go to workers.
# The default of 2 reproduces the old server cap if the notebook is run by hand.
N_JOBS      = int(os.environ.get('KI_N_JOBS', 2))
MIN_CLASS_N = 20

# Live progress (notebook output is only written when the notebook finishes).
PROGRESS = RESULTS_DIR / 'logs' / 'nb03_progress_v2.txt'
PROGRESS.parent.mkdir(parents=True, exist_ok=True)
def progress(msg):
    with open(PROGRESS, 'a') as fh:
        fh.write(f"{time.strftime('%H:%M:%S')}  {msg}\n")
PROGRESS.write_text('')

# Smoke-run cap. NB03_MAX_TARGETS limits the benchmark to the first N targets so a
# shake-out run reaches every downstream notebook in minutes. Unset (the default)
# runs every target. run_pipeline.py records these knobs in its manifest, so a capped
# run can never be mistaken for -- or skipped in favour of -- a full one.
_max_t = os.environ.get('NB03_MAX_TARGETS')
if _max_t:
    _n = int(_max_t)
    # Keep pre-specified compounds in the sample. NB08/NB09's producer-evidence
    # path only runs when NB03 benchmarks a pre-specified compound, so a smoke run
    # of DA targets alone gives false confidence about exactly those notebooks
    # (the first smoke run capped to three DA targets and NB08 then crashed).
    _pre_set = set(pa_features) | set(dc_features)
    _pre  = [t for t in target_metabolites if t in _pre_set]
    _rest = [t for t in target_metabolites if t not in _pre_set]
    _k = min(len(_pre), max(1, _n // 2))
    print(f'SMOKE RUN: target list capped at {_n} of {len(target_metabolites)} '
          f'({_k} pre-specified + {min(_n - _k, len(_rest))} DA)')
    target_metabolites = _pre[:_k] + _rest[:_n - _k]

# Permutation budget, set once the target list is final. Sequential p-values: a target
# stops at its PERM_H-th exceedance (p = PERM_H / L), so null-like targets cost ~20-100
# permutations; otherwise it runs to N_PERM and p = (x + 1) / (N_PERM + 1). N_PERM >=
# 20 x targets lets one target pass BH at 0.05 among them; a fixed 100 permutations
# could not (q floor ~ m / 101). NB03_N_PERM overrides the cap.
_n_perm_env = os.environ.get('NB03_N_PERM')
N_PERM = int(_n_perm_env) if _n_perm_env else max(1000, 20 * len(target_metabolites))
print(f'Permutation test: sequential, stops at {PERM_H} exceedances, at most {N_PERM} '
      f'permutations per target (p floor {1 / (N_PERM + 1):.4f})')

# -- timing probe: project the total runtime before committing to it --
# CAVEAT: this times a FIXED-parameter fit, but the nested tuning explores costlier
# settings, so the projection is a lower bound -- it predicted 2.75 h for a run
# that took ~36 h on the old server. Treat it as a floor, not an estimate.
if target_metabolites:
    _X, _y, _fc, _ = build_feature_matrix(spc, mtb, target_metabolites[0], conf_df)
    _yb = binarize_target(_y)
    _cache = build_fold_cache(_X, _yb, n_splits=N_OUTER)
    _t0 = time.time(); run_cv_auc_cached(_cache, _yb); _t_fit = (time.time() - _t0) / N_OUTER
    _fits_per_target = N_OUTER * (N_ITER * N_INNER + 1) + N_OUTER * (N_PERM + 1)
    _proj_h = len(target_metabolites) * _fits_per_target * _t_fit / N_JOBS / 3600
    print(f'Timing probe: {_t_fit:.2f} s/fit -> projected {_proj_h:.2f} h for '
          f'{len(target_metabolites)} targets ({_fits_per_target} fits each, {N_JOBS} workers)')
    progress(f'probe {_t_fit:.2f} s/fit (fixed params), projected {_proj_h:.2f} h, '
             f'{len(target_metabolites)} targets, N_PERM={N_PERM}')
    if _proj_h > 2.75:
        print('WARNING: projected NB03 runtime exceeds 2.75 h')

bench_rows = []
roc_store  = {}
tuned_params_by_target = {}
skipped = []

for col_name in target_metabolites:
    if col_name not in mtb.columns:
        print(f'  SKIP (not in MTB): {col_name}')
        skipped.append((col_name, 'absent from reduced matrix')); continue

    X, y, feat_cols, _ = build_feature_matrix(spc, mtb, col_name, conf_df)
    y_bin = binarize_target(y)
    pos_n, neg_n = int(y_bin.sum()), int(len(y_bin) - y_bin.sum())
    if min(pos_n, neg_n) < MIN_CLASS_N:
        print(f'  SKIP (minority class < {MIN_CLASS_N}, ties at median): {col_name}  '
              f'pos={pos_n}, neg={neg_n}')
        skipped.append((col_name, f'minority class {min(pos_n, neg_n)} < {MIN_CLASS_N}'))
        continue

    # Performance: nested CV
    nested_aucs, chosen = nested_cv_auc(X, y_bin, n_outer=N_OUTER, n_iter=N_ITER,
                                        inner=N_INNER, random_state=42, n_jobs=N_JOBS)
    tuned_params_by_target[col_name] = chosen

    # Significance: fixed-parameter observed vs permuted, identical folds
    fold_cache = build_fold_cache(X, y_bin, n_splits=N_OUTER, random_state=42)
    mean_fpr, mean_tpr, fixed_aucs = run_cv_auc_cached(fold_cache, y_bin, None, random_state=42)
    roc_store[col_name] = (mean_fpr, mean_tpr, fixed_aucs)
    fixed_auc = float(fixed_aucs.mean())
    # Sequential: permutation i always uses seed 1000 + i, so where it stops does not
    # depend on the batch size or the number of workers.
    perm_p, n_perm_run, n_exceed, perm_early, perm_aucs = utils.sequential_permutation_p(
        fixed_auc,
        lambda a, b: run_permutation_null_cached(fold_cache, y_bin, None, n_perm=b - a,
                                                 random_state=42, base_seed=1000 + a,
                                                 n_jobs=N_JOBS),
        h=PERM_H, b_max=N_PERM, batch_size=max(16, 2 * N_JOBS))

    # What do the confounders explain on their own? R2 of the raw target, and the
    # fixed-parameter AUC of a confounders-only model: for the RAW split (what the old
    # test silently credited to the species) and for the ADJUSTED split (~0.5 when the
    # residualisation removed them; well above that means nonlinear confounding the
    # linear adjustment missed).
    y_raw, Z = confounder_view(spc, mtb, col_name, conf_df)
    r2_conf = 1.0 - float(np.var(y)) / float(np.var(y_raw))
    raw_bin = binarize_target(y_raw)
    auc_conf_raw = (float(run_cv_auc_cached(build_fold_cache(Z, raw_bin, n_splits=N_OUTER),
                                            raw_bin)[2].mean())
                    if min(raw_bin.sum(), len(raw_bin) - raw_bin.sum()) >= N_OUTER else np.nan)
    auc_conf_adj = float(run_cv_auc_cached(build_fold_cache(Z, y_bin, n_splits=N_OUTER),
                                           y_bin)[2].mean())

    in_e, in_a = col_name in top15_early_set, col_name in top15_advanced_set
    stage_tag = 'Both' if in_e and in_a else 'Early' if in_e else 'Advanced' if in_a else 'none'
    # Target_type names EVERY criterion this target met. It used to fall through to
    # 'robust_DA' for anything that was not a polyamine, which labelled the 9 targets
    # contributed only by the decarboxylation class as robust DA hits -- tryptamine
    # among them, whose DA q-values are 0.873 and 0.971. The column is the provenance
    # claim that defends the class against target-selection circularity, so it is
    # derived from the selection sets, and a target that matches none is a bug.
    _tags = ['robust_DA'] if (in_e or in_a) else []
    if col_name in pa_features:
        _tags.append('prespecified_polyamine')
    if col_name in dc_features:
        _tags.append('prespecified_decarboxylation')
    assert _tags, f'{col_name} is benchmarked under no admissible selection criterion'
    target_type = '+'.join(_tags)

    bench_rows.append({
        'Metabolite':          col_name,
        'Short_name':          short_name(col_name),
        'Classifier':          'XGBoost',
        'Target_type':         target_type,
        'AUC_nested':          float(nested_aucs.mean()),
        'AUC_nested_std':      float(nested_aucs.std()),
        'AUC_fixed':           fixed_auc,
        'N':                   len(y_bin),
        'N_high':              pos_n,
        'N_low':               neg_n,
        'is_polyamine':        col_name in POLYAMINE_COL_NAMES,
        'CRC_stage':           stage_tag,
        'Direction':           direction_map.get(col_name, 'not DA-selected'),
        'Target_definition':   'median split of the target residualised on the confounders',
        'Perm_statistic':      'AUC_fixed',
        'Perm_pvalue':         float(perm_p),
        'Perm_null_AUC_mean':  float(perm_aucs.mean()),
        'N_perm':              n_perm_run,
        'N_perm_max':          N_PERM,
        'Perm_exceedances':    n_exceed,
        'Perm_stopped_early':  perm_early,
        'R2_confounders':      r2_conf,
        'AUC_confounders_raw': auc_conf_raw,
        'AUC_confounders_adjusted': auc_conf_adj,
        'Nested_params':       chosen,
    })
    progress(f'{len(bench_rows)}/{len(target_metabolites)} {short_name(col_name)[:40]} '
             f'AUC_nested={nested_aucs.mean():.3f} AUC_fixed={fixed_auc:.3f} perm_p={perm_p:.4f}')
    print(f'  {short_name(col_name):40s} n={len(y_bin):3d}  AUC_nested={nested_aucs.mean():.3f}'
          f'±{nested_aucs.std():.3f}  AUC_fixed={fixed_auc:.3f}  perm_p={perm_p:.4f} ({n_perm_run} perms)  '
          f'(null AUC={perm_aucs.mean():.3f})  confounders alone: R2={r2_conf:.3f}, '
          f'AUC raw={auc_conf_raw:.3f} / adjusted={auc_conf_adj:.3f}')

bench_df = pd.DataFrame(bench_rows)
if not bench_df.empty:
    bench_df['Perm_qvalue'] = multipletests(bench_df['Perm_pvalue'], method='fdr_bh')[1]
    # backwards-compatible aliases used by the plots/tables below
    bench_df['AUC_mean'] = bench_df['AUC_nested']
    bench_df['AUC_std']  = bench_df['AUC_nested_std']
    n_q05 = int((bench_df['Perm_qvalue'] < 0.05).sum())
    print(f'\nBenchmark done: {len(bench_df)} metabolites modelled, {n_q05} with Perm_qvalue < 0.05')
    print('Every p-value tests the species GIVEN the confounders (target residualised on them).')
    print(f"Confounders alone -- raw split: median AUC {bench_df['AUC_confounders_raw'].median():.3f} "
          f"(max {bench_df['AUC_confounders_raw'].max():.3f}); adjusted split: max AUC "
          f"{bench_df['AUC_confounders_adjusted'].max():.3f}")
    _nonlin = bench_df.loc[bench_df['AUC_confounders_adjusted'] > 0.60, 'Short_name'].tolist()
    if _nonlin:
        print(f'  Possible nonlinear confounding (confounders still predict the adjusted '
              f'split at AUC > 0.60): {_nonlin}')
else:
    print('\nBenchmark produced no rows -- every target was skipped.')
if skipped:
    print(f'Skipped {len(skipped)} target(s):')
    for name, why in skipped:
        print(f'  {short_name(name)[:45]:45s} {why}')

In [ ]:
# ── Save benchmark table + summary bar chart ─────────────────────────────────
bench_df.to_csv(RESULTS_DIR / 'tables' / 'ml_benchmark_dysregulated_metabolites_v2.csv',
                 index=False)
print(f'Saved: ml_benchmark_dysregulated_metabolites_v2.csv  ({len(bench_df)} rows)')

if not bench_df.empty:
    order = bench_df.sort_values('AUC_mean', ascending=False)
    colors = ['#c0392b' if pa else '#3498db' for pa in order['is_polyamine']]
    fig, ax = plt.subplots(figsize=(max(10, len(order) * 0.5 + 2), 5))
    bars = ax.bar(order['Short_name'], order['AUC_mean'], yerr=order['AUC_std'],
                   color=colors, capsize=3, alpha=0.85)
    for bar, sig in zip(bars, order['Perm_qvalue'] < 0.05):
        if sig:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                    '*', ha='center', fontsize=14, fontweight='bold')
    ax.axhline(0.5, color='gray', lw=1.5, ls='--', label='Chance (AUC=0.5)')
    ax.set_xticklabels(order['Short_name'], rotation=45, ha='right', fontsize=8)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel('AUC (nested CV, mean \u00b1 sd)', fontsize=11)
    ax.set_title('ML Benchmark — XGBoost per target metabolite (nested-CV AUC)\n'
                  'Red = polyamine | * = permutation q < 0.05 (BH across targets)', fontsize=11, fontweight='bold')
    ax.legend(loc='upper right', fontsize=9)
    plt.tight_layout()
    out = RESULTS_DIR / 'figures' / 'ml' / 'benchmark_auc_bars_v2.png'
    viz.save_figure(plt.gcf(), out)
    plt.show(); plt.close()
    print(f'Saved: {out}')


In [ ]:
# ── AUC summary table (difficulty annotation) ────────────────────────────────
if not bench_df.empty:
    summary_rows = []
    for _, row in bench_df.iterrows():
        difficulty = ('predictable' if row['AUC_mean'] >= 0.75 else
                      'moderate'    if row['AUC_mean'] >= 0.60 else
                      'low signal')
        summary_rows.append({
            'Metabolite':   row['Metabolite'],
            'Short_name':   row['Short_name'],
            'AUC_mean':     round(row['AUC_mean'], 3),
            'AUC_std':      round(row['AUC_std'], 3),
            'Perm_pvalue':  round(row['Perm_pvalue'], 4),
            'Perm_qvalue':  round(row['Perm_qvalue'], 4),
            'Target_type':  row['Target_type'],
            'Direction':    row['Direction'],
            'CRC_stage':    row['CRC_stage'],
            'is_polyamine': row['is_polyamine'],
            'Difficulty':   difficulty,
        })
    auc_summary = pd.DataFrame(summary_rows).sort_values('AUC_mean', ascending=False)
    print('=== ML AUC Summary ===')
    print(auc_summary[['Short_name', 'Target_type', 'AUC_mean', 'Perm_pvalue', 'Perm_qvalue',
                        'Direction', 'Difficulty']].to_string(index=False))
    print('\nDifficulty breakdown:')
    print(auc_summary['Difficulty'].value_counts().to_string())
else:
    auc_summary = pd.DataFrame()
    print('No benchmark rows — auc_summary is empty.')


---
## Section 3: Species attribution by OOF SHAP — **every benchmarked target** (v2.2)

For each target benchmarked in Section 2, using the **fixed-parameter** model (the same
model whose significance was tested — never one tuned on the explained data):

1. **In-sample SHAP** (`shap_store`) — labelled diagnostic only.
2. **Out-of-fold SHAP** (`oof_shap_store`) — each sample explained by a model that never saw
   it. This is the ranking source for `species_attribution`.
3. **Direction**: Spearman ρ between each species' (scaled) abundance and its OOF SHAP value
   for "high metabolite" (high *for its confounders*: the target is adjusted).
   `Direction = positive` means more of the species pushes the prediction towards high
   metabolite levels — only those are candidate producers downstream.
4. Bootstrap 95% CI on each top species' rank (resampling rows of the OOF |SHAP| matrix).

**v2.2 — the polyamine gate is gone.** This loop used to iterate the 7 pre-specified
polyamines only, so `oof_shap_store` held 7 compounds while Section 2 benchmarked 27. The
predictive evidence source in NB08/NB09 could therefore only ever nominate a polyamine
producer, whatever the metabolome-wide statistics said. It now covers every benchmarked
target; `Is_polyamine` flags the pre-specified subset, which is reported, not privileged.

`Model_perm_q` (the target's permutation q-value) is carried into every row: if the model
itself is not better than chance, its SHAP ranking is not evidence of anything. **Figures
are drawn only for targets that beat their permutation null**, for the same reason — a
ranked bar chart of a chance-level model invites over-reading.

`pa_producers` is still written, as the polyamine-only slice of this table, because NB08's
R3/R6 read it under that name.

In [ ]:
# ── Polyamines: FDR-dysregulated (descriptive) vs pre-specified (analysed) ──
pa_early    = top15_early[top15_early['is_polyamine']].copy()
pa_advanced = top15_advanced[top15_advanced['is_polyamine']].copy()

for df, label in [(pa_early, 'Early CRC'), (pa_advanced, 'Advanced CRC')]:
    df['PA_name'] = df['Feature'].map(KEGG_TO_NAME)
    print(f'\n{label}: FDR-significant polyamines among the selected DA targets ({len(df)}):')
    if df.empty:
        print('  None')
    else:
        print(df[['PA_name', 'Feature', 'CliffsD', 'QValue']].to_string(index=False))

# The producer analysis runs on every pre-specified polyamine that was benchmarked.
pa_union = [c for c in pa_targets['Feature'].tolist()] if not pa_targets.empty else []
print(f'\nPre-specified polyamines for producer analysis: {len(pa_union)}')
print(pa_union)


In [ ]:
# ── SHAP computation helpers (fixed-parameter model) ─────────────────────────

def compute_insample_shap(X, y_bin, random_state=42):
    """Fit on the FULL cohort and explain the same samples. Diagnostic only."""
    pipe = make_pipeline(None, random_state=random_state)
    pipe.fit(X, y_bin)
    X_pre = pipe[:-1].transform(X)                   # every step but the classifier
    explainer = shap.TreeExplainer(pipe.named_steps['clf'])
    return extract_shap_class1(explainer.shap_values(X_pre))


def compute_oof_shap(X, y_bin, n_splits=5, random_state=42):
    """Out-of-fold SIGNED SHAP plus the matching preprocessed feature values."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    oof = np.zeros(X.shape, dtype=np.float64)
    oof_X = np.zeros(X.shape, dtype=np.float64)
    for train_idx, test_idx in skf.split(X, y_bin):
        pipe = make_pipeline(None, random_state=random_state)
        pipe.fit(X[train_idx], y_bin[train_idx])
        X_test_pre = pipe[:-1].transform(X[test_idx])
        explainer = shap.TreeExplainer(pipe.named_steps['clf'])
        oof[test_idx] = extract_shap_class1(explainer.shap_values(X_test_pre))
        oof_X[test_idx] = X_test_pre
    return oof, oof_X


def shap_direction(oof_X, oof_sv, idx):
    """Spearman rho between a feature's value and its SHAP value (sign = direction)."""
    rhos = []
    for j in idx:
        if np.std(oof_X[:, j]) == 0 or np.std(oof_sv[:, j]) == 0:
            rhos.append(np.nan)
        else:
            rhos.append(spearmanr(oof_X[:, j], oof_sv[:, j])[0])
    return np.array(rhos)


def bootstrap_shap_rank_ci(oof_shap_abs, spc_idx, spc_names, top_n=20, n_boot=1000, random_state=42):
    """Bootstrap 95% CI on species rank, resampling ROWS of the OOF |SHAP| matrix."""
    boot_rng = np.random.default_rng(random_state)
    n_samples = oof_shap_abs.shape[0]
    spc_shap = oof_shap_abs[:, spc_idx]
    mean_abs = spc_shap.mean(axis=0)
    order = np.argsort(mean_abs)[::-1][:top_n]
    top_names = [spc_names[i] for i in order]
    top_mean = mean_abs[order]

    boot_ranks = np.zeros((n_boot, len(order)))
    for b in range(n_boot):
        idx_b = boot_rng.integers(0, n_samples, size=n_samples)
        resampled_mean = spc_shap[idx_b].mean(axis=0)
        full_ranks = rankdata(-resampled_mean, method='average')  # rank 1 = highest
        boot_ranks[b] = full_ranks[order]

    ci_low  = np.percentile(boot_ranks, 2.5, axis=0)
    ci_high = np.percentile(boot_ranks, 97.5, axis=0)
    stable  = (ci_high - ci_low) <= 10   # CI width <= 10 ranks = stable
    return order, top_names, top_mean, ci_low, ci_high, stable


print('SHAP helpers defined: compute_insample_shap, compute_oof_shap, shap_direction, bootstrap_shap_rank_ci.')


In [ ]:
# ── Per-target species attribution (every benchmarked target, v2.2) ──────────
spc_cols_set = set(c for c in spc.columns if c != 'Sample')
bench_idx = bench_df.set_index('Metabolite') if not bench_df.empty else pd.DataFrame()

shap_store        = {}   # in-sample SHAP -- diagnostic only, NOT used for ranking
oof_shap_store    = {}   # OOF SHAP -- ranking source
attribution_rows  = []
N_BOOT = 1000

# Every target Section 2 actually benchmarked, not just the polyamine subset. pa_union
# (the 7 pre-specified polyamines) now only decides the Is_polyamine flag.
targets_for_shap = list(bench_df['Metabolite']) if not bench_df.empty else []
_pa_set = set(pa_union)
print(f'SHAP attribution over {len(targets_for_shap)} benchmarked target(s); '
      f'{len(_pa_set & set(targets_for_shap))} of them pre-specified polyamines')

for col_name in targets_for_shap:
    mtb_name    = KEGG_TO_NAME.get(col_name, species_short(col_name))
    mtb_name_fs = (mtb_name.lower().replace(' ', '_').replace('/', '_')
                   .replace(',', '').replace('(', '').replace(')', ''))
    is_pa = col_name in _pa_set
    print(f'\n--- {mtb_name} ({col_name}){"  [pre-specified polyamine]" if is_pa else ""} ---')

    if bench_idx.empty or col_name not in bench_idx.index:
        print('  Not benchmarked in Section 2 (e.g. skipped for class imbalance) — skip'); continue
    model_q = float(bench_idx.loc[col_name, 'Perm_qvalue'])
    model_auc = float(bench_idx.loc[col_name, 'AUC_nested'])

    X, y, feat_cols, _ = build_feature_matrix(spc, mtb, col_name, conf_df)
    y_bin = binarize_target(y)

    spc_idx   = [i for i, f in enumerate(feat_cols) if f in spc_cols_set]
    spc_names = [feat_cols[i] for i in spc_idx]

    insample_sv = compute_insample_shap(X, y_bin, random_state=42)
    shap_store[col_name] = {'feature_names': feat_cols,
                            'mean_abs_shap': np.abs(insample_sv).mean(axis=0),
                            'pa_name': mtb_name,
                            'is_polyamine': is_pa}

    oof_sv, oof_X = compute_oof_shap(X, y_bin, n_splits=N_OUTER, random_state=42)
    oof_abs = np.abs(oof_sv)
    dir_rho_all = shap_direction(oof_X, oof_sv, range(len(feat_cols)))
    oof_shap_store[col_name] = {'feature_names': feat_cols,
                                'oof_mean_abs_shap': oof_abs.mean(axis=0),
                                'oof_mean_signed_shap': oof_sv.mean(axis=0),
                                'shap_direction_rho': dir_rho_all,
                                'pa_name': mtb_name,
                                'is_polyamine': is_pa}

    rho, rho_p = spearmanr(oof_abs.mean(axis=0)[spc_idx], shap_store[col_name]['mean_abs_shap'][spc_idx])
    flag = '  [in-sample looks inflated relative to OOF]' if rho < 0.7 else ''
    print(f'  Model: AUC_nested={model_auc:.3f}, permutation q={model_q:.4f}')
    print(f'  In-sample vs OOF species-SHAP rank agreement: rho={rho:.3f} (p={rho_p:.3g}){flag}')

    order, top_names, top_mean, ci_low, ci_high, stable = bootstrap_shap_rank_ci(
        oof_abs, spc_idx, spc_names, top_n=20, n_boot=N_BOOT, random_state=42)

    for rank, (o, sp, val, cil, cih, st) in enumerate(
            zip(order, top_names, top_mean, ci_low, ci_high, stable), 1):
        j = spc_idx[o]
        d_rho = dir_rho_all[j]
        attribution_rows.append({
            'Metabolite':        mtb_name,
            'Metabolite_column': col_name,
            'Is_polyamine':      is_pa,
            'Species':           sp,
            'Mean_SHAP_OOF':     float(val),          # mean |SHAP| (importance)
            'Mean_SHAP_signed':  float(oof_sv[:, j].mean()),
            'SHAP_dir_rho':      float(d_rho) if np.isfinite(d_rho) else np.nan,
            'Direction':         ('positive' if d_rho > 0 else 'negative') if np.isfinite(d_rho) else 'undetermined',
            'Rank':              rank,
            'CI_lower':          float(cil),
            'CI_upper':          float(cih),
            'Rank_stable':       bool(st),
            'Model_AUC_nested':  model_auc,
            'Model_perm_q':      model_q,
        })
        if rank <= 10:
            print(f'    {rank:2d}. {species_short(sp):40s}  OOF |SHAP|={val:.4f}  direction rho={d_rho:+.2f}')

    # Figure only where the model beat its permutation null: a ranked bar chart of a
    # chance-level model reads as a finding when it is not one. The table keeps every row.
    if not (np.isfinite(model_q) and model_q < utils.FDR_THRESHOLD):
        print(f'  Figure skipped: model perm q={model_q:.3f} does not beat chance '
              f'(q<{utils.FDR_THRESHOLD}); the ranking is not interpretable.')
        continue

    display_names = [species_short(s) for s in top_names]
    dirs = [dir_rho_all[spc_idx[o]] for o in order]
    fig, ax = plt.subplots(figsize=(8, max(5, 0.45 * len(top_names) + 1.5)))
    colors_s = [('#c0392b' if d > 0 else '#2471a3') if st else '#95a5a6' for st, d in zip(stable, dirs)]
    ax.barh(range(len(top_names)), top_mean, color=colors_s, alpha=0.85)
    ax.set_yticks(range(len(top_names)))
    ax.set_yticklabels(display_names, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel('Mean OOF |SHAP|', fontsize=10)
    ax.set_title(f'{mtb_name} — Top 20 species by OOF SHAP (model perm. q={model_q:.3f})\n'
                 f'red = stable rank & positive direction | blue = stable & negative | grey = unstable rank',
                 fontsize=10, fontweight='bold')
    plt.tight_layout()
    out_shap = RESULTS_DIR / 'figures' / 'ml' / f'attribution_{mtb_name_fs}_oof_shap_v2.png'
    viz.save_figure(plt.gcf(), out_shap)
    plt.show(); plt.close()
    print(f'  Saved: {out_shap}')

# Explicit schema. With nothing benchmarked, attribution_rows is empty, and a bare
# DataFrame() of an empty list has NO columns -- so every downstream reader that selects
# columns raised KeyError instead of handling 'no attribution'. The column list must
# match the dict keys above.
ATTRIBUTION_COLUMNS = ['Metabolite', 'Metabolite_column', 'Is_polyamine', 'Species',
                       'Mean_SHAP_OOF', 'Mean_SHAP_signed', 'SHAP_dir_rho', 'Direction',
                       'Rank', 'CI_lower', 'CI_upper', 'Rank_stable', 'Model_AUC_nested',
                       'Model_perm_q']
attribution_df = pd.DataFrame(attribution_rows, columns=ATTRIBUTION_COLUMNS)
attribution_df.to_csv(RESULTS_DIR / 'tables' / 'species_attribution_shap_v2.csv', index=False)

# Back-compat view: NB08's R3/R6 read `pa_producers` with the old column names.
PA_PRODUCER_COLUMNS = ['Polyamine', 'Polyamine_column', 'Species', 'Mean_SHAP_OOF',
                       'Mean_SHAP_signed', 'SHAP_dir_rho', 'Direction', 'Rank',
                       'CI_lower', 'CI_upper', 'Rank_stable', 'Model_AUC_nested',
                       'Model_perm_q']
pa_producer_df = (attribution_df[attribution_df['Is_polyamine']]
                  .rename(columns={'Metabolite': 'Polyamine',
                                   'Metabolite_column': 'Polyamine_column'})
                  .reindex(columns=PA_PRODUCER_COLUMNS)
                  .reset_index(drop=True))

print(f'\nAttribution rows built from OOF SHAP: {len(attribution_df)} '
      f'over {attribution_df["Metabolite_column"].nunique()} target(s)')
if not attribution_df.empty:
    print(attribution_df['Direction'].value_counts().to_string())
    _n_interp = int((attribution_df['Model_perm_q'] < utils.FDR_THRESHOLD).sum())
    print(f'  rows whose model beat its permutation null: {_n_interp} '
          f'({_n_interp / len(attribution_df):.0%})')
print(f'  polyamine-only back-compat view (pa_producers): {len(pa_producer_df)} rows')
print('  saved: tables/species_attribution_shap_v2.csv')

In [ ]:
# ── Save producer candidates + cross-polyamine heatmap ───────────────────────
# The CSV is written UNCONDITIONALLY: output_contract declares it with min_rows=0, so an
# empty table is fine but a missing one is a contract failure. It used to be written only
# when a polyamine was benchmarked, so a run whose pre-specified sample happened to hold
# none produced no file -- the same defect already fixed for r8.
pa_producer_df.to_csv(RESULTS_DIR / 'tables' / 'polyamine_producer_candidates_v2.csv',
                      index=False)
print(f'Saved: polyamine_producer_candidates_v2.csv  ({len(pa_producer_df)} rows'
      + ('' if not pa_producer_df.empty else ' -- no polyamine was benchmarked') + ')')

# The heatmap stays conditional: a figure of nothing is not worth drawing, and its
# contract entry allows min_files=0.
if not pa_producer_df.empty:
    top_spc_all = (pa_producer_df.groupby('Species')['Mean_SHAP_OOF']
                   .max().nlargest(20).index.tolist())
    pivot_pa = (pa_producer_df[pa_producer_df['Species'].isin(top_spc_all)]
                .pivot_table(index='Species', columns='Polyamine',
                             values='Mean_SHAP_OOF', aggfunc='mean')
                .fillna(0))
    pivot_pa = pivot_pa.reindex(top_spc_all, fill_value=0)
    display_labels = [species_short(s) for s in pivot_pa.index]

    fig, ax = plt.subplots(figsize=(max(6, len(pivot_pa.columns) * 2.5),
                                     max(6, len(top_spc_all) * 0.5 + 2)))
    sns.heatmap(pivot_pa, annot=True, fmt='.3g', cmap='YlOrRd', linewidths=0.4, ax=ax,
                cbar_kws={'label': 'Mean OOF |SHAP|'})
    ax.set_yticklabels(display_labels, fontsize=8)
    ax.set_title('SHAP-ranked species for pre-specified polyamines\n'
                 'OOF |SHAP| importance (fixed-parameter XGBoost | YACHIDA cohort)',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('Polyamine', fontsize=10)
    ax.set_ylabel('Species', fontsize=10)
    plt.tight_layout()
    out_hm = RESULTS_DIR / 'figures' / 'ml' / 'polyamine_producer_heatmap_v2.png'
    viz.save_figure(plt.gcf(), out_hm)
    plt.show(); plt.close()
    print(f'Saved: {out_hm}')

    top1 = pa_producer_df[pa_producer_df['Rank'] == 1].copy()
    top1['Species_short'] = top1['Species'].apply(species_short)
    print('\nTop producer candidate per polyamine (OOF SHAP rank 1):')
    print(top1[['Polyamine', 'Species_short', 'Mean_SHAP_OOF', 'Direction', 'Model_perm_q']].to_string(index=False))
else:
    print('No pre-specified polyamine was benchmarked — empty table written, no heatmap.')

---
## Save ML Results

This notebook stops here — no cross-notebook evidence-triangulation step (that logic
now lives at the end of `09_pathway_network.ipynb`, where NB06's and NB08's `_v2`
outputs genuinely exist; see fix #7 above).


In [ ]:
ml_results = {
    'top15_early':                top15_early,
    'top15_advanced':             top15_advanced,
    'benchmark_df':                bench_df,
    'auc_summary':                 auc_summary,
    'dysreg_polyamines_early':    pa_early,
    'dysreg_polyamines_advanced': pa_advanced,
    # v2.2: the metabolome-wide attribution table is the predictive evidence source.
    # `pa_producers` is its polyamine-only slice, kept because NB08's R3/R6 read it.
    'species_attribution':         attribution_df,
    'pa_producers':                pa_producer_df,
    'oof_shap_store':              oof_shap_store,
    'shap_store':                  shap_store,
    'polyamine_targets':           pa_targets,
    'nested_params':               tuned_params_by_target,
    'fixed_params':                FIXED_PARAMS,
    'n_perm':                      N_PERM,
}

with open(RESULTS_DIR / 'intermediate' / 'ml_results_v2.pkl', 'wb') as f:
    pickle.dump(ml_results, f, protocol=pickle.HIGHEST_PROTOCOL)

print('ML results saved to ml_results_v2.pkl')
print(f'  Benchmark: {len(bench_df)} metabolites')
print(f'  FDR-dysregulated polyamines — Early: {len(pa_early)} | Advanced: {len(pa_advanced)}')
print(f'  Pre-specified polyamine targets: {len(pa_targets)}')
print(f'  Species attribution (OOF SHAP): {len(attribution_df)} rows over '
      f'{attribution_df["Metabolite_column"].nunique() if not attribution_df.empty else 0} target(s)')
print(f'    of which the polyamine slice (pa_producers): {len(pa_producer_df)} rows')
if oof_shap_store:
    print(f'  OOF SHAP stored for {len(oof_shap_store)} target(s): '
          f'{sorted(v["pa_name"] for v in oof_shap_store.values())}')